# 08 — HRSC2016: training a ship detector

Your first real training run. HRSC2016 is small (about a thousand harbour images, one class: `ship`), has an honest held-out test split, and trains in hours instead of days. This lesson explores the data, explains why its images are resized differently from DOTA tiles, measures the published checkpoint on the test split, and runs a short training job whose checkpoint you then evaluate the same way.

| | |
|---|---|
| Kaggle input | [`guofeng/hrsc2016`](https://www.kaggle.com/datasets/guofeng/hrsc2016) |
| Local | `HRSC_DATA_ROOT` → folder containing `FullDataSet/` and `ImageSets/` |
| GPU | Recommended; required for the training run (`RUN_SMOKE = True`) |
| Recipes | `oriented_rcnn/hrsc2016_le90_3x.json` (published), `..._1x.json` (shorter) |
| Published (held-out test mAP50) | Oriented R-CNN **90.41%**, Rotated Faster R-CNN **88.77%**, Rotated FCOS **88.34%** |
| Post | [HRSC2016 recipes, trains, and results](https://deeplearning.earth/posts/2026-09-13_hrsc2016_recipes_trains_and_results/) |

## The HRSC2016 protocol

- **Data:** 1,061 Google Earth images of harbours and coastlines, 300 to 1,500 px on a side. Standard split: `trainval` for training, `test` for evaluation. The test images are never trained on, so the number you measure is the number you can quote (unlike DOTA eval-val, lesson 07).
- **Recipe:** MMRotate's protocol — a ResNet-50 initialised from ImageNet (not from DOTA), 36 epochs ("3×"), flips, and random rotation of up to ±20° in the 3× recipes.
- **Resizing:** `keep_ratio` to a long edge of 800 px, padded to a multiple of 32, instead of DOTA's fixed 1024 canvas. Section 4 shows why.

## Setup

The same two cells open every lesson. The first installs OrientedDet: your editable checkout if this notebook sits inside the repo, otherwise the PyPI wheel. The second sets paths and the device, and defines the small helpers used across the series (`hub_files`, `hub_model`, `detect`, `show`, ...). On Kaggle, switch **Internet ON** in the notebook settings.

In [ ]:
# Install: editable checkout when this notebook lives inside the repo, else PyPI.
import importlib.util
from pathlib import Path

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file() and (p / "oriented_det").is_dir()), None)
_spec = importlib.util.find_spec("oriented_det")
if REPO is None:
    %pip install -q "oriented-det>=0.3.1"
elif _spec is None or not Path(_spec.origin).resolve().is_relative_to(REPO.resolve()):
    %pip install -q -e "{REPO}"

In [ ]:
import os, subprocess, sys, time, urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import oriented_det

GH_RAW = "https://raw.githubusercontent.com/DL4EO/oriented-det/main"
IN_KAGGLE = Path("/kaggle/working").is_dir()
HAS_REPO = REPO is not None
WORK = (Path("/kaggle/working") if IN_KAGGLE else
        Path(os.environ.get("ORIENTED_DET_TUTORIAL_WORK", (REPO or Path.cwd()) / "work_dirs")).resolve())
WORK.mkdir(parents=True, exist_ok=True)
os.environ["ORIENTED_DET_PROJECT_ROOT"] = str(WORK)  # runs/ and predictions/ land here
if not HAS_REPO:  # wheel install: keep Hub weights out of site-packages
    os.environ.setdefault("ORIENTED_DET_PRETRAINED_DIR", str(WORK / "pretrained"))

CONFIGS = Path(oriented_det.__file__).parent / "configs"  # recipes shipped with the package
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
ODET = [sys.executable, "-m", "oriented_det.cli"]  # same as typing `odet` in a shell


def run(cmd):
    """Run a CLI command, echoing it first."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(["odet", *cmd[3:]] if cmd[:3] == ODET else cmd), flush=True)
    subprocess.run(cmd, check=True)


def demo_file(rel):
    """demo/<rel> from the checkout, else downloaded once from GitHub into WORK/demo."""
    if HAS_REPO and (REPO / "demo" / rel).is_file():
        return REPO / "demo" / rel
    target = WORK / "demo" / rel
    if not target.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{GH_RAW}/demo/{rel}", target)
    return target


def kaggle_input(*needles):
    """First /kaggle/input/<dataset> whose name contains one of `needles` (None off Kaggle)."""
    root = Path("/kaggle/input")
    dirs = sorted(p for p in root.iterdir() if p.is_dir()) if root.is_dir() else []
    return next((p for n in needles for p in dirs if n.lower() in p.name.lower()), None)


def hub_files(slug):
    """Download a Hub checkpoint and its sidecar config once -> (checkpoint_path, config_path)."""
    from huggingface_hub import hf_hub_download
    from oriented_det.pretrained import ensure_checkpoint, load_manifest, resolve_checkpoint_sidecar_config

    ckpt = ensure_checkpoint(f"hf://{slug}")
    manifest = load_manifest()
    sidecar = resolve_checkpoint_sidecar_config(ckpt) or hf_hub_download(
        manifest["repo_id"], ckpt.with_suffix(".json").name,
        revision=manifest.get("revision", "main"), local_dir=ckpt.parent)
    return ckpt, Path(sidecar)


def hub_model(slug, device=DEVICE):
    """Hub checkpoint -> (model, config, class_names). Lesson 01 unpacks it."""
    from oriented_det.runtime.checkpoint import load_model_from_checkpoint

    ckpt, sidecar = hub_files(slug)
    return load_model_from_checkpoint(str(ckpt), str(sidecar), device=device)


def detect(model, config, class_names, image, score_thr=None, classes=None):
    """Detections on a PIL image with the checkpoint's deploy settings (sliding windows past the canvas)."""
    from oriented_det.runtime.inference import apply_nms_to_detections, run_inference_auto
    from oriented_det.train.config import (get_preprocessing_params, resolve_inference_score_threshold,
                                           resolve_inference_sliding_window_overlap_pixels)

    nms = config.production.final_nms_iou_threshold
    dets = run_inference_auto(
        image=image, model=model, device=DEVICE, preprocessing=get_preprocessing_params(config),
        score_threshold=resolve_inference_score_threshold(config) if score_thr is None else score_thr,
        nms_threshold=nms, overlap_pixels=resolve_inference_sliding_window_overlap_pixels(config),
        class_names=class_names)
    dets = apply_nms_to_detections(dets, iou_threshold=nms)
    if classes:
        dets = [d for d in dets if class_names[int(d["label"]) - 1] in classes]
    return dets


def show(*images, titles=(), size=6):
    """Display images side by side."""
    _, axes = plt.subplots(1, len(images), figsize=(size * len(images), size), squeeze=False)
    for ax, img, title in zip(axes[0], images, [*titles, *[""] * len(images)]):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print(f"oriented_det: {Path(oriented_det.__file__).parent}")
print(f"device={DEVICE}  kaggle={IN_KAGGLE}  work={WORK}")

## 1. Point at the data

In [ ]:
from oriented_det.data.hrsc2016 import HRSC2016Dataset

RUN_SMOKE = False  # True: a short training run on a GPU (a few minutes)

DATA_ROOT = kaggle_input("hrsc2016", "hrsc") or Path(os.environ.get("HRSC_DATA_ROOT", "/path/to/data/HRSC2016"))
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"HRSC2016 not found at {DATA_ROOT}. On Kaggle add the dataset guofeng/hrsc2016; "
                            "locally set HRSC_DATA_ROOT to the folder containing FullDataSet/ and ImageSets/.")
train = HRSC2016Dataset(DATA_ROOT, split="trainval", difficult_strategy="keep")
test = HRSC2016Dataset(DATA_ROOT, split="test", difficult_strategy="keep")
print(f"trainval {len(train)} images, test {len(test)} images, classes {train.get_class_names()}")

## 2. What the images and ships look like

In [ ]:
import numpy as np

sizes = np.array([(s.width, s.height) for s in train])
ships = [a.rbox for s in train for a in s.annotations]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(sizes[:, 0], sizes[:, 1], s=6, alpha=0.5)
axes[0].set_xlabel("width (px)"); axes[0].set_ylabel("height (px)"); axes[0].set_title("image sizes")
axes[1].hist([len(s.annotations) for s in train], bins=range(0, 30)); axes[1].set_title("ships per image")
axes[2].hist([max(r.width, r.height) / min(r.width, r.height) for r in ships], bins=40)
axes[2].set_title("ship aspect ratio (long / short side)")
plt.tight_layout(); plt.show()
print(f"{len(ships)} training ships, median length {np.median([max(r.width, r.height) for r in ships]):.0f} px")

Ships are long and thin (aspect ratios of 4 to 8 are common), so a few degrees of angle error costs a lot of IoU. This is where oriented boxes and angle-aware losses earn their keep.

In [ ]:
from PIL import Image
from oriented_det.utils.viz import DrawingSpec, draw_boxes

chips = []
for s in list(train)[:3]:
    img = Image.open(s.image_path).convert("RGB")
    chips.append(draw_boxes(img, [a.rbox for a in s.annotations],
                            specs=[DrawingSpec(outline=(0, 255, 128), width=3)] * len(s.annotations)))
show(*chips, titles=[s.image_path.name for s in list(train)[:3]], size=5)

## 3. Fixed canvas vs keep-ratio

DOTA tiles are all exactly 1024×1024, so `resize_mode: fixed` does nothing to them. HRSC images come in many sizes and aspect ratios. Stretching them onto a square canvas would squash ships and change their apparent angle, so the HRSC recipes use `keep_ratio`: scale until the long edge is 800 px, then pad the right and bottom to a multiple of 32, so every FPN stride divides the image evenly. The recipes also switch to ImageNet normalisation, matching the ImageNet backbone they start from.

Whole-image resizing also means there is no sliding window at inference (lesson 06): `run_inference_auto` rescales the full image, exactly as in training.

In [ ]:
from oriented_det.train.config import TrainingExperimentConfig, get_preprocessing_params

dota = get_preprocessing_params(TrainingExperimentConfig.load(CONFIGS / "oriented_rcnn/dota_le90_1x.json"))
hrsc_cfg = TrainingExperimentConfig.load(CONFIGS / "oriented_rcnn/hrsc2016_le90_3x.json")
hrsc = get_preprocessing_params(hrsc_cfg)
print(f"{'':16s} {'DOTA 1x':34s} HRSC 3x")
for key in ("resize_mode", "target_size", "normalize_mean", "normalize_std"):
    print(f"{key:16s} {str(dota.get(key)):34s} {hrsc.get(key)}")
print(f"{'random rotate':16s} {'off':34s} +/-{hrsc_cfg.preprocessing.random_rotate_angle_range} deg, "
      f"p={hrsc_cfg.preprocessing.random_rotate_prob}")

img = Image.open(train[0].image_path).convert("RGB")
w, h = img.size
scale = 800 / max(w, h)
nw, nh = round(w * scale), round(h * scale)
padded = Image.new("RGB", (-(-nw // 32) * 32, -(-nh // 32) * 32))
padded.paste(img.resize((nw, nh)), (0, 0))
show(img.resize((1024, 1024)), padded,
     titles=(f"fixed: {w}x{h} stretched to 1024x1024", f"keep_ratio: {nw}x{nh}, padded to {padded.width}x{padded.height}"))

## 4. How good is the published checkpoint?

Evaluate `oriented_rcnn_hrsc2016_le90_3x` on the test split with the same recipe as lesson 07: keep every detection down to 0.05, then compute mAP50. On a GPU this runs the whole test split (about a minute); on CPU, set `N_TEST` to a small number.

In [ ]:
from oriented_det.data import Detection, GroundTruth, compute_oriented_map


def evaluate(model, config, class_names, samples, keep_class="ship"):
    """mAP50 of one class over dataset samples, keeping every detection down to score 0.05."""
    dets, gts = {}, {}
    for s in samples:
        gts[s.image_path.stem] = [GroundTruth(a.rbox, 0, keep_class, a.difficult, s.image_path.stem) for a in s.annotations]
        found = detect(model, config, class_names, Image.open(s.image_path).convert("RGB"), score_thr=0.05, classes=[keep_class])
        dets[s.image_path.stem] = [Detection(d["rbox"], float(d["score"]), 0, keep_class, s.image_path.stem) for d in found]
    return compute_oriented_map(dets, gts, class_names=[keep_class], show_progress=False)[0]

In [ ]:
from oriented_det.runtime.inference import visualize_results

N_TEST = None if DEVICE == "cuda" else 40  # None = the whole test split
subset = list(test)[:N_TEST]
model, config, class_names = hub_model("oriented_rcnn_hrsc2016_le90_3x")
t0 = time.perf_counter()
hub_map = evaluate(model, config, class_names, subset)
print(f"Hub 3x checkpoint: mAP50 {100 * hub_map:.2f}% on {len(subset)} test images "
      f"({time.perf_counter() - t0:.0f} s). Published on all test images: 90.41%")

s = subset[0]
img = Image.open(s.image_path).convert("RGB")
gt = draw_boxes(img.copy(), [a.rbox for a in s.annotations], specs=[DrawingSpec(outline=(0, 255, 128), width=3)] * len(s.annotations))
show(gt, visualize_results(img.copy(), detect(model, config, class_names, img), class_names),
     titles=("labels", f"predictions @ deploy floor {config.production.score_threshold}"))

The deploy floor for this checkpoint is 0.85: with a single class and a two-stage head, scores are very peaked (lesson 04).

## 5. Train it yourself

A full 3× run takes several hours on one GPU. The smoke run below uses the 1× recipe, 32 training images, and one epoch: enough to exercise the whole pipeline (data loading, augmentation, loss, checkpointing), not enough to learn much. Its checkpoint is then scored with the same `evaluate()` as the Hub model; starting from ImageNet weights, expect an mAP near zero.

In [ ]:
SMOKE_RECIPE = CONFIGS / "oriented_rcnn/hrsc2016_le90_1x.json"
cfg = TrainingExperimentConfig.load(SMOKE_RECIPE)
cfg.dataset.data_root = DATA_ROOT
cfg.dataset.max_train_samples = 32
cfg.dataset.max_val_samples = 16
cfg.training.num_epochs = 1
cfg.evaluation.compute_map_every_n_epochs = 1
smoke = WORK / "08_hrsc_smoke.json"
cfg.save(smoke)
print("wrote", smoke)

In [ ]:
def newest_run(model_type):
    """(checkpoint, config) of the most recent odet train run for this model type."""
    run_dir = max((WORK / "runs" / model_type).iterdir(), key=lambda p: p.stat().st_mtime)
    ckpt = max((run_dir / "checkpoints").glob("*.pth"), key=lambda p: p.stat().st_mtime)
    return ckpt, run_dir / "config.json"

In [ ]:
import gc

import torch
from oriented_det.runtime.checkpoint import load_model_from_checkpoint

if not RUN_SMOKE:
    print("RUN_SMOKE=False: skipping training. Set it to True in section 1 on a GPU.")
else:
    model = None  # hand the GPU memory to the training subprocess
    gc.collect(); torch.cuda.empty_cache()
    run([*ODET, "train", "--config", smoke])
    ckpt, run_config = newest_run("oriented_rcnn")
    mine = load_model_from_checkpoint(str(ckpt), str(run_config), device=DEVICE)
    print(f"smoke checkpoint {ckpt.name}: mAP50 {100 * evaluate(*mine, subset):.2f}% "
          f"vs Hub 3x {100 * hub_map:.2f}% on the same {len(subset)} images")

## 6. The full recipe

```bash
odet train --config oriented_det/configs/oriented_rcnn/hrsc2016_le90_3x.json   # set dataset.data_root first
make eval-val EXPERIMENT=runs/oriented_rcnn/<timestamp>                         # held-out test, in a checkout
```

| Hub checkpoint | Held-out test mAP50 |
|---|---:|
| `oriented_rcnn_hrsc2016_le90_3x` | 90.41% |
| `rotated_faster_rcnn_hrsc2016_le90_3x` | 88.77% |
| `rotated_fcos_hrsc2016_le90_3x` | 88.34% |

## What to try next

- Start from DOTA instead of ImageNet: set `cfg.checkpoint.load_from_checkpoint = "hf://oriented_rcnn_dota_le90_1x"` (the 15-class head is re-initialised for one class) and compare the first epochs.
- Evaluate `rotated_fcos_hrsc2016_le90_3x` with `evaluate()`: is the gap to Oriented R-CNN the same on your subset?
- `odet hrsc-to-dota` exports HRSC to DOTA-format folders if you want to reuse DOTA tooling.
- **Next:** [09 — FAIR1M fine-grained finetune](09_fair1m_finetune.ipynb).